# Bike Demand Time-Series Workflow

This notebook implements the full `fetch -> prepare -> analyze -> report` workflow for the bike-demand workshop dataset. It is a complete worked example that writes the same style of stage artifacts used elsewhere in the repo, but it does the full hourly-profile and report work directly inside the notebook.


In [1]:
from __future__ import annotations

import json
import os
import shutil
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root from the current working directory.")


ROOT = find_repo_root()
os.chdir(ROOT)
NEWLINE = chr(10)

WORKFLOW_ROOT = ROOT / "runs" / "notebook-bike-demand-analysis"
FETCH_DIR = WORKFLOW_ROOT / "fetch"
PREPARE_DIR = WORKFLOW_ROOT / "prepare"
ANALYZE_DIR = WORKFLOW_ROOT / "analyze"
REPORT_DIR = WORKFLOW_ROOT / "report"

for directory in [FETCH_DIR, PREPARE_DIR, ANALYZE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_CSV = ROOT / "data" / "source" / "bike_demand_source.csv"
RAW_CSV = FETCH_DIR / "bike_demand_raw.csv"
PREPARED_CSV = PREPARE_DIR / "prepared_demand.csv"
DATASET_OVERVIEW_JSON = ANALYZE_DIR / "dataset_overview.json"
HOURLY_PROFILE_CSV = ANALYZE_DIR / "hourly_demand_profile.csv"
DAILY_CYCLE_PNG = ANALYZE_DIR / "weekday_weekend_daily_cycle.png"
REPORT_MD = REPORT_DIR / "report.md"

HOURLY_PROFILE_COLUMNS = [
    "day_type",
    "hour",
    "n_observations",
    "mean_demand",
    "median_demand",
    "std_demand",
]

print(f"Repo root: {ROOT}")
print(f"Notebook workflow output: {WORKFLOW_ROOT}")


Repo root: /home/can134/nextgen/nextgen2026-coding-bootcamp
Notebook workflow output: /home/can134/nextgen/nextgen2026-coding-bootcamp/runs/notebook-bike-demand-analysis


## Fetch

The fetch stage materializes the raw bike-demand dataset into a stable CSV artifact so later steps can read from disk instead of from the source file directly.


In [2]:
shutil.copy2(SOURCE_CSV, RAW_CSV)
raw_table = pd.read_csv(RAW_CSV)

fetch_artifacts = {
    "dataset_name": "bike_rental_demand",
    "raw_csv": str(RAW_CSV),
    "n_rows": int(len(raw_table)),
    "columns": raw_table.columns.tolist(),
}

raw_table.head()


       dteday  hr  cnt  season weathersit
0  2024-04-04   0   15  spring      clear
1  2024-04-04   1   10  spring      clear
2  2024-04-04   2    8  spring      clear
3  2024-04-04   3    6  spring      clear
4  2024-04-04   4    7  spring      clear

## Prepare

The prepare stage standardizes the time column, derives `hour` and `day_type`, sorts rows chronologically, and writes a clean analysis-ready demand table.


In [3]:
timestamp = pd.to_datetime(raw_table["dteday"]) + pd.to_timedelta(raw_table["hr"].astype(int), unit="h")

prepared = pd.DataFrame(
    {
        "timestamp": timestamp,
        "demand": raw_table["cnt"].astype(int),
        "hour": timestamp.dt.hour.astype(int),
        "day_type": timestamp.dt.dayofweek.map(lambda day: "weekday" if day < 5 else "weekend"),
        "season": raw_table["season"].astype(str),
        "weather": raw_table["weathersit"].astype(str),
    }
)
prepared = prepared.sort_values("timestamp", ignore_index=True)
prepared["timestamp"] = prepared["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S")
prepared.to_csv(PREPARED_CSV, index=False)

prepare_artifacts = {
    "prepared_csv": str(PREPARED_CSV),
    "n_rows": int(len(prepared)),
    "columns": prepared.columns.tolist(),
    "day_types": sorted(prepared["day_type"].unique().tolist()),
}

prepared.head()


             timestamp  demand  hour day_type  season weather
0  2024-04-04T00:00:00      15     0  weekday  spring   clear
1  2024-04-04T01:00:00      10     1  weekday  spring   clear
2  2024-04-04T02:00:00       8     2  weekday  spring   clear
3  2024-04-04T03:00:00       6     3  weekday  spring   clear
4  2024-04-04T04:00:00       7     4  weekday  spring   clear

## Analyze

The analyze stage computes a dataset overview, an hourly demand profile split by weekday versus weekend, and a daily-cycle plot. This is the full implementation of the workshop task rather than the scaffolded starter-state version.


In [4]:
def build_dataset_overview(prepared_df: pd.DataFrame) -> dict:
    timestamps = pd.to_datetime(prepared_df["timestamp"])
    day_type_counts = prepared_df["day_type"].value_counts().sort_index()
    return {
        "dataset_name": "bike_rental_demand",
        "n_rows": int(len(prepared_df)),
        "timestamp_start": timestamps.min().strftime("%Y-%m-%dT%H:%M:%S"),
        "timestamp_end": timestamps.max().strftime("%Y-%m-%dT%H:%M:%S"),
        "target_column": "demand",
        "day_types": sorted(prepared_df["day_type"].unique().tolist()),
        "rows_per_day_type": {key: int(value) for key, value in day_type_counts.items()},
    }


def build_hourly_demand_profile(prepared_df: pd.DataFrame) -> pd.DataFrame:
    profile = (
        prepared_df.groupby(["day_type", "hour"], as_index=False)
        .agg(
            n_observations=("demand", "size"),
            mean_demand=("demand", "mean"),
            median_demand=("demand", "median"),
            std_demand=("demand", lambda values: values.std(ddof=0)),
        )
        .sort_values(["day_type", "hour"], ignore_index=True)
    )
    return profile[HOURLY_PROFILE_COLUMNS]


def write_daily_cycle_plot(profile_df: pd.DataFrame, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(9, 5))
    palette = {"weekday": "#1f5aa6", "weekend": "#d17a22"}
    for day_type in ["weekday", "weekend"]:
        subset = profile_df.loc[profile_df["day_type"] == day_type]
        ax.plot(
            subset["hour"],
            subset["mean_demand"],
            marker="o",
            linewidth=2.2,
            color=palette[day_type],
            label=day_type.title(),
        )
    ax.set_title("Bike Demand Across the Daily Cycle")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Mean Demand")
    ax.set_xticks(range(24))
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)


dataset_overview = build_dataset_overview(prepared)
hourly_profile = build_hourly_demand_profile(prepared)

DATASET_OVERVIEW_JSON.write_text(json.dumps(dataset_overview, indent=2) + NEWLINE)
hourly_profile.to_csv(HOURLY_PROFILE_CSV, index=False)
write_daily_cycle_plot(hourly_profile, DAILY_CYCLE_PNG)

analyze_artifacts = {
    "dataset_overview_json": str(DATASET_OVERVIEW_JSON),
    "hourly_demand_profile_csv": str(HOURLY_PROFILE_CSV),
    "weekday_weekend_daily_cycle_png": str(DAILY_CYCLE_PNG),
}

hourly_profile.head(8)


  day_type  hour  n_observations  mean_demand  median_demand  std_demand
0  weekday     0               2         15.0           15.0         0.0
1  weekday     1               2         10.0           10.0         0.0
2  weekday     2               2          8.0            8.0         0.0
3  weekday     3               2          6.0            6.0         0.0
4  weekday     4               2          7.0            7.0         0.0
5  weekday     5               2         13.0           13.0         1.0
6  weekday     6               2         32.0           32.0         2.0
7  weekday     7               2         74.0           74.0         4.0

## Report

The report stage reads the analyze artifacts, builds an `Hourly Demand Profiles` section from `hourly_demand_profile.csv`, and writes a Markdown report that references the daily-cycle plot instead of recomputing anything.


In [5]:
def format_profile_for_markdown(profile_df: pd.DataFrame) -> pd.DataFrame:
    formatted = profile_df.copy()
    formatted["hour"] = formatted["hour"].astype(int).astype(str)
    formatted["n_observations"] = formatted["n_observations"].astype(int).astype(str)
    for column in ["mean_demand", "median_demand", "std_demand"]:
        formatted[column] = formatted[column].map(lambda value: f"{value:.4f}")
    return formatted


def dataframe_to_markdown_table(df: pd.DataFrame) -> str:
    columns = list(df.columns)
    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = [
        "| " + " | ".join(str(row[column]) for column in columns) + " |"
        for _, row in df.iterrows()
    ]
    return NEWLINE.join([header, separator, *rows])


def build_hourly_demand_profiles_section(hourly_profile_csv: Path) -> list[str]:
    profile_df = pd.read_csv(hourly_profile_csv)
    profile_table = dataframe_to_markdown_table(format_profile_for_markdown(profile_df))
    return [
        "## Hourly Demand Profiles",
        "",
        "The table below is rendered directly from `hourly_demand_profile.csv` written by the analyze stage.",
        "",
        profile_table,
    ]


profile_section = build_hourly_demand_profiles_section(HOURLY_PROFILE_CSV)
plot_relpath = Path("../analyze") / DAILY_CYCLE_PNG.name

report_lines = [
    "# Bike Demand Workflow Report",
    "",
    "This report was generated from analyze-stage artifacts created inside the notebook workflow.",
    "",
    "## Dataset Overview",
    "",
    f"- Dataset: `{dataset_overview['dataset_name']}`",
    f"- Rows analyzed: `{dataset_overview['n_rows']}`",
    f"- Time range: `{dataset_overview['timestamp_start']}` to `{dataset_overview['timestamp_end']}`",
    f"- Target column: `{dataset_overview['target_column']}`",
    f"- Day types: `{', '.join(dataset_overview['day_types'])}`",
    f"- Rows per day type: `{dataset_overview['rows_per_day_type']}`",
    "",
    "## Analyze Artifacts",
    "",
    f"- `{DATASET_OVERVIEW_JSON.name}`",
    f"- `{HOURLY_PROFILE_CSV.name}`",
    f"- `{DAILY_CYCLE_PNG.name}`",
    "",
    "## Daily Demand Cycle",
    "",
    "The chart below is referenced from the analyze-stage PNG written by the notebook workflow.",
    "",
    f"![Weekday versus weekend bike demand]({plot_relpath.as_posix()})",
    "",
    *profile_section,
]

REPORT_MD.write_text(NEWLINE.join(report_lines) + NEWLINE)

report_artifacts = {
    "report_markdown": str(REPORT_MD),
}

print(REPORT_MD.read_text())


# Bike Demand Workflow Report

This report was generated from analyze-stage artifacts created inside the notebook workflow.

## Dataset Overview

- Dataset: `bike_rental_demand`
- Rows analyzed: `96`
- Time range: `2024-04-04T00:00:00` to `2024-04-07T23:00:00`
- Target column: `demand`
- Day types: `weekday, weekend`
- Rows per day type: `{'weekday': 48, 'weekend': 48}`

## Analyze Artifacts

- `dataset_overview.json`
- `hourly_demand_profile.csv`
- `weekday_weekend_daily_cycle.png`

## Daily Demand Cycle

The chart below is referenced from the analyze-stage PNG written by the notebook workflow.

![Weekday versus weekend bike demand](../analyze/weekday_weekend_daily_cycle.png)

## Hourly Demand Profiles

The table below is rendered directly from `hourly_demand_profile.csv` written by the analyze stage.

| day_type | hour | n_observations | mean_demand | median_demand | std_demand |
| --- | --- | --- | --- | --- | --- |
| weekday | 0 | 2 | 15.0000 | 15.0000 | 0.0000 |
| weekday | 1 | 2 

## Validation

The final cell checks the end-to-end contract and lists the artifacts written by the notebook workflow.


In [6]:
assert RAW_CSV.exists()
assert PREPARED_CSV.exists()
assert DATASET_OVERVIEW_JSON.exists()
assert HOURLY_PROFILE_CSV.exists()
assert DAILY_CYCLE_PNG.exists()
assert REPORT_MD.exists()

validated_profile = pd.read_csv(HOURLY_PROFILE_CSV)
assert list(validated_profile.columns) == HOURLY_PROFILE_COLUMNS
assert sorted(validated_profile["day_type"].unique().tolist()) == ["weekday", "weekend"]
assert validated_profile.shape[0] == 48

report_text = REPORT_MD.read_text()
assert "## Hourly Demand Profiles" in report_text
assert "hourly_demand_profile.csv" in report_text
assert "## Daily Demand Cycle" in report_text

print("Notebook workflow artifacts:")
for path in sorted(WORKFLOW_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(ROOT))

validated_profile.head(12)


Notebook workflow artifacts:
runs/notebook-bike-demand-analysis/analyze/dataset_overview.json
runs/notebook-bike-demand-analysis/analyze/hourly_demand_profile.csv
runs/notebook-bike-demand-analysis/analyze/weekday_weekend_daily_cycle.png
runs/notebook-bike-demand-analysis/fetch/bike_demand_raw.csv
runs/notebook-bike-demand-analysis/prepare/prepared_demand.csv
runs/notebook-bike-demand-analysis/report/report.md


   day_type  hour  n_observations  mean_demand  median_demand  std_demand
0   weekday     0               2         15.0           15.0         0.0
1   weekday     1               2         10.0           10.0         0.0
2   weekday     2               2          8.0            8.0         0.0
3   weekday     3               2          6.0            6.0         0.0
4   weekday     4               2          7.0            7.0         0.0
5   weekday     5               2         13.0           13.0         1.0
6   weekday     6               2         32.0           32.0         2.0
7   weekday     7               2         74.0           74.0         4.0
8   weekday     8               2        114.0          114.0         4.0
9   weekday     9               2         92.5           92.5         2.5
10  weekday    10               2         62.0           62.0         2.0
11  weekday    11               2         56.5           56.5         1.5